# Observability & Evaluation

*Level 7 — Production RAG*

## Objective

Run a real evaluation batch against the **live, running API** (retrieval quality + hand-rolled
Ragas-style faithfulness/answer-relevance), save it as this level's tracked baseline, and check
future runs against it with `production_eval/regression_suite.py`. Same prerequisite as Notebook
01: the API (port 8001) and the docker-compose stack must already be running.

In [1]:
import json
import sys
import time
from pathlib import Path

import requests

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))

from production_common.dataset import prepare
from production_common.embed import OllamaEmbedder
from production_common.llm import OllamaLLM
from production_eval.ragas_eval import answer_relevance, faithfulness
from production_eval.retrieval_eval import mrr, ndcg_at_k, recall_at_k

BASE_URL = "http://127.0.0.1:8001"
API_KEY = "dev-local-key"

data = prepare()
print(f"corpus: {len(data.corpus)} docs, questions: {len(data.questions)}")

corpus: 300 docs, questions: 600


## Sample a fresh, seeded batch of real questions

Deterministic (Python's own `random.Random(seed)`, not the dataset's own shuffle) so this is
reproducible across runs, and distinct from the individual questions already exercised manually
in Notebooks 01-02 and `examples/production_app/client.py`.

In [2]:
import random

EVAL_SEED = 7
N_EVAL = 8

rng = random.Random(EVAL_SEED)
qids = rng.sample(list(data.questions.keys()), N_EVAL)
eval_questions = [{"qid": qid, **data.questions[qid]} for qid in qids]

for q in eval_questions[:3]:
    print(q["question"], "->", q["answers"][:1])

What is the only episode released on VCD? -> ['The Infinite Quest']
Who did BSkyB compete with initially? -> ['ONdigital']
Which player did the Panthers lose to an ACL injury in a preseason game? -> ['Kelvin Benjamin']


## Run each question through the real, live API

Real HTTP calls, real retrieval, real generation -- this is the actual production path, not an
in-process shortcut. Each response's `sources` gives us the retrieved doc_ids for the retrieval
metrics; `answer` feeds the faithfulness/relevance judge calls below.

In [3]:
results = {}
latencies_ms = []

for q in eval_questions:
    t0 = time.perf_counter()
    response = requests.post(
        f"{BASE_URL}/query",
        json={"question": q["question"], "top_k": 5},
        headers={"x-api-key": API_KEY},
        timeout=120,
    )
    elapsed_ms = (time.perf_counter() - t0) * 1000
    body = response.json()
    results[q["qid"]] = {
        "question": q["question"],
        "gold_doc_id": q["gold_doc_id"],
        "gold_answers": q["answers"],
        "answer": body["answer"],
        "sources": body["sources"],
        "cache_hit": body["cache_hit"],
        "latency_ms": elapsed_ms,
    }
    latencies_ms.append(elapsed_ms)
    print(f"[{'HIT ' if body['cache_hit'] else 'MISS'}] {elapsed_ms:7.0f}ms  {q['question'][:70]}")

print(f"\ndone: {len(results)} questions, {sum(1 for r in results.values() if not r['cache_hit'])} cache misses")

[MISS]    5258ms  What is the only episode released on VCD?


[MISS]    3595ms  Who did BSkyB compete with initially?


[MISS]    3258ms  Which player did the Panthers lose to an ACL injury in a preseason gam


[MISS]    5051ms  When was the study on sequenced Y genomes published?


[MISS]    6132ms  What does the First Company Law Directive article 11 require?


[MISS]    2915ms  Ludwig Krapf recorded the name was what?


[MISS]    3584ms  Of what mountain system are the Victorian Alps a part?


[MISS]    3213ms  Where did the Panthers practice for the Super Bowl?

done: 8 questions, 8 cache misses


## Retrieval quality: Recall@5, MRR, NDCG@5

Same metrics as Level 2's evaluation, reimplemented in `production_eval/retrieval_eval.py`.

In [4]:
retrieved = {qid: [s["doc_id"] for s in r["sources"]] for qid, r in results.items()}
qrels = {qid: {r["gold_doc_id"]} for qid, r in results.items()}

recall5 = recall_at_k(retrieved, qrels, k=5)
mrr_score = mrr(retrieved, qrels)
ndcg5 = ndcg_at_k(retrieved, qrels, k=5)

print(f"recall@5 = {recall5:.3f}")
print(f"mrr      = {mrr_score:.3f}")
print(f"ndcg@5   = {ndcg5:.3f}")

recall@5 = 1.000
mrr      = 0.938
ndcg@5   = 0.954


## Answer quality: hand-rolled Ragas-style faithfulness + answer relevance

Two separate judge calls per question (`production_eval/ragas_eval.py`, Ollama-based, not the
real `ragas` package -- see its module docstring): does every claim in the answer trace back to
the retrieved context, and does the answer actually address the question asked.

In [5]:
llm = OllamaLLM()
embedder = OllamaEmbedder()

faithfulness_scores = []
relevance_scores = []

for qid, r in results.items():
    context = "\n\n".join(f"{s['title']}: " for s in r["sources"])  # placeholder, replaced below
    # Reconstruct the actual context text from the corpus (the API only returns titles/scores,
    # not full chunk text, in its `sources` field -- so pull the real indexed text back out of
    # the corpus by doc_id for the judge, matching exactly what the API itself retrieved).
    context = "\n\n".join(data.corpus[s["doc_id"]]["text"] for s in r["sources"] if s["doc_id"] in data.corpus)

    f_result = faithfulness(r["answer"], context, llm=llm)
    r_result = answer_relevance(r["question"], r["answer"], llm=llm, embedder=embedder)

    results[qid]["faithfulness"] = f_result
    results[qid]["answer_relevance"] = r_result
    faithfulness_scores.append(f_result["score"])
    relevance_scores.append(r_result["score"])
    print(f"faithfulness={f_result['score']:.2f}  relevance={r_result['score']:.2f}  {r['question'][:60]}")

avg_faithfulness = sum(faithfulness_scores) / len(faithfulness_scores)
avg_relevance = sum(relevance_scores) / len(relevance_scores)
print(f"\naverage faithfulness   = {avg_faithfulness:.3f}")
print(f"average answer_relevance = {avg_relevance:.3f}")

faithfulness=1.00  relevance=0.44  What is the only episode released on VCD?


faithfulness=1.00  relevance=0.87  Who did BSkyB compete with initially?


faithfulness=0.00  relevance=0.69  Which player did the Panthers lose to an ACL injury in a pre


faithfulness=0.75  relevance=0.71  When was the study on sequenced Y genomes published?


faithfulness=0.50  relevance=0.81  What does the First Company Law Directive article 11 require


faithfulness=0.00  relevance=0.51  Ludwig Krapf recorded the name was what?


faithfulness=1.00  relevance=0.81  Of what mountain system are the Victorian Alps a part?


faithfulness=0.00  relevance=0.85  Where did the Panthers practice for the Super Bowl?

average faithfulness   = 0.531
average answer_relevance = 0.711


## Investigating the lowest-scoring case(s)

An aggregate score alone can't tell you *why* -- print the actual context, answer, and per-claim
judgments for the worst-scoring question(s), the same way earlier levels traced individual CRAG /
Self-RAG / verification-agent judgments back to their raw evidence.

In [6]:
worst_qid = min(results, key=lambda k: results[k]["faithfulness"]["score"])
worst = results[worst_qid]

print("QUESTION:", worst["question"])
print("GOLD ANSWER(S):", worst["gold_answers"])
print("\nMODEL ANSWER:", worst["answer"])
print("\nFAITHFULNESS SCORE:", worst["faithfulness"]["score"])
print("\nPER-CLAIM JUDGMENTS:")
for claim in worst["faithfulness"]["claims"]:
    print(f"  [{'SUPPORTED' if claim['supported'] else 'NOT SUPPORTED'}] {claim['claim']}")
print("\nRETRIEVED CONTEXT (first 1500 chars):")
context = "\n\n".join(data.corpus[s["doc_id"]]["text"] for s in worst["sources"] if s["doc_id"] in data.corpus)
print(context[:1500])

QUESTION: Which player did the Panthers lose to an ACL injury in a preseason game?
GOLD ANSWER(S): ['Kelvin Benjamin', 'Kelvin Benjamin', 'Benjamin']

MODEL ANSWER: Kelvin Benjamin was lost to a torn ACL in the preseason.

FAITHFULNESS SCORE: 0.0

PER-CLAIM JUDGMENTS:
  [NOT SUPPORTED] Kelvin Benjamin suffered an injury
  [NOT SUPPORTED] The injury was a torn ACL
  [NOT SUPPORTED] It occurred during the preseason

RETRIEVED CONTEXT (first 1500 chars):
Despite waiving longtime running back DeAngelo Williams and losing top wide receiver Kelvin Benjamin to a torn ACL in the preseason, the Carolina Panthers had their best regular season in franchise history, becoming the seventh team to win at least 15 regular season games since the league expanded to a 16-game schedule in 1978. Carolina started the season 14–0, not only setting franchise records for the best start and the longest single-season winning streak, but also posting the best start to a season by an NFC team in NFL history, break

## What I observed

**Retrieval is excellent and stable**: recall@5 = 1.000, mrr = 0.938, ndcg@5 = 0.954 -- consistent
with the original 15-question baseline (recall@5 = 1.0, mrr = 0.933, ndcg@5 = 0.951). Every
question's gold document was retrieved, on a fresh, differently-seeded 8-question sample.

**A textbook-clear case of judge unreliability, caught live in this exact run**: the Kelvin
Benjamin question. The model's answer -- *"Kelvin Benjamin was lost to a torn ACL in the
preseason"* -- is almost a verbatim match of the retrieved context's own sentence: *"...losing top
wide receiver Kelvin Benjamin to a torn ACL in the preseason..."*. The retrieved context plainly
supports every word. **The faithfulness judge marked all three extracted claims "NOT SUPPORTED"
anyway, scoring this answer 0.0/1.0** -- the worst possible score for what is, by direct
inspection, a fully-grounded answer. This is not a subtle edge case: the claim and the context
share almost identical wording, and the judge (`llama3.2`, the same 3B local model used for
generation) still failed to connect them. It's the same lesson as Level 4's CRAG grading, Level
5's source-checking, and Level 6's verification agent -- **an LLM judge has its own error rate,
and that error rate is not small enough to trust blindly, even on an easy case.**

**The regression suite genuinely flagged a regression** -- and it's informative about the
regression suite's own limits, not just the model's: `answer_relevance` dropped from the 15-question
baseline's 0.788 to this run's 0.711 (a 0.077 drop, just over the 0.05 tolerance), while
`faithfulness` actually *rose* (0.375 -> 0.531, likely because this smaller sample happened to
draw fewer hard cases like the one above, or literally is affected by the same judge noise in the
opposite direction). **A flat point-comparison against a single small baseline run is noisy: an
8-question and a 15-question sample of *different* real questions will never produce identical
scores even with an unchanged system.** A real deployment of this regression suite would want
either a large, fixed eval set (hundreds of questions, not 8-15) or a statistical tolerance band
around repeated-baseline variance, not a single point value -- a genuine, evidenced limitation of
`production_eval/regression_suite.py` as currently built, not something to paper over.

**What this run did *not* touch**: `baseline_metrics.json` was left as the original 15-question
run (a larger, more stable reference sample); only `last_run_metrics.json` was overwritten with
this run's numbers, exactly as a real "most recent run vs. tracked baseline" workflow would do.

## Saving this run and checking for regressions

In [7]:
from production_eval.regression_suite import check_regression, load_baseline, save_baseline

current_metrics = {
    "recall_at_5": recall5,
    "mrr": mrr_score,
    "ndcg_at_5": ndcg5,
    "faithfulness": avg_faithfulness,
    "answer_relevance": avg_relevance,
}

last_run_file = LEVEL_DIR / "production_eval" / "last_run_metrics.json"
last_run_file.write_text(json.dumps(current_metrics, indent=2))

baseline = load_baseline()
print("baseline:", baseline)
print("current: ", current_metrics)

outcome = check_regression(current_metrics, baseline=baseline)
print("\nregression check:", outcome["status"])
if outcome["regressions"]:
    for reg in outcome["regressions"]:
        print(" -", reg)

baseline: {'recall_at_5': 1.0, 'mrr': 0.9333333333333333, 'ndcg_at_5': 0.9507906338095277, 'faithfulness': 0.3749206349206349, 'answer_relevance': 0.7879370809853348}
current:  {'recall_at_5': 1.0, 'mrr': 0.9375, 'ndcg_at_5': 0.9538662191964322, 'faithfulness': 0.53125, 'answer_relevance': 0.7112077902024624}

regression check: regression
 - {'metric': 'answer_relevance', 'baseline': 0.7879370809853348, 'current': 0.7112077902024624, 'drop': 0.07672929078287238}
